# DataCleanEnv — GRPO Training on Kaggle

Train an LLM to clean messy CSV data using GRPO (Group Relative Policy Optimization).

## Kaggle Setup
1. **Enable GPU**: Settings → Accelerator → GPU P100
2. **Enable Internet**: Settings → Internet → On
3. Run all cells in order

## Hardware
- **P100 (16GB VRAM)**: Can train Qwen2.5-3B with 4-bit quantization
- **2x T4 (if available)**: Can train larger models

## 1. Clone Repository & Install Dependencies

In [ ]:
# Clone the repository
!git clone https://github.com/YOUR_USERNAME/openenv-datacleaning-agent.git
%cd openenv-datacleaning-agent

# Install dependencies
!pip install -q openenv-core[core] transformers accelerate bitsandbytes peft trl datasets
!pip install -q fastapi fastmcp uvicorn pandas numpy
!pip install -e ./data_clean_env

print("✅ Dependencies installed")

## 2. GPU Check & Configuration

In [ ]:
import torch
import os
import gc

# ============================================
# KAGGLE OOM FIX: Aggressive memory management
# ============================================
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# Check GPU
print("=" * 50)
print("GPU STATUS")
print("=" * 50)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu_name}")
    print(f"✅ VRAM: {gpu_memory:.1f} GB")
else:
    print("❌ No GPU! Enable GPU in Settings → Accelerator")
    raise RuntimeError("GPU required")

# ============================================
# USE 1.5B MODEL FOR KAGGLE (prevents OOM)
# 3B model crashes Kaggle kernel even with P100
# ============================================
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # SMALLER = STABLE
print(f"\n📦 Selected model: {MODEL_ID}")
print(f"   (Using 1.5B to prevent Kaggle kernel crash)")

# ============================================
# 🔑 HUGGINGFACE TOKEN
# ============================================
HF_TOKEN = ""  # <-- Paste your token here, or use Kaggle Secrets

if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        print("✅ HF_TOKEN loaded from Kaggle Secrets")
    except:
        print("⚠️ No HF_TOKEN - downloads may be slower")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("✅ Logged in to HuggingFace")

# ============================================
# KAGGLE-OPTIMIZED CONFIG (prevents OOM crash)
# ============================================
CONFIG = {
    "learning_rate": 2e-5,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,  # Larger to compensate for small batch
    "max_completion_length": 256,      # REDUCED from 384 (saves memory)
    "num_generations": 1,              # REDUCED from 2 (biggest memory saver!)
    "beta": 0.1,
    "temperature": 0.7,
    "output_dir": "./dataclean-grpo-kaggle",
    "logging_steps": 1,
    "save_steps": 50,
}

print(f"\n⚡ Kaggle-optimized config:")
print(f"   num_generations: {CONFIG['num_generations']} (1 = less memory)")
print(f"   max_completion_length: {CONFIG['max_completion_length']}")
print(f"   gradient_accumulation: {CONFIG['gradient_accumulation_steps']}")
print(f"\n⏱️ Estimated training time: ~30-45 min on P100")

## 3. Generate Training Data (Using Same Script as Colab)

In [ ]:
# Generate diverse training data using the same script as Colab
!python generate_training_data.py --output-dir data_clean_env/tasks --num-samples 50 --num-datasets 5

# List generated files
import os
tasks_dir = "data_clean_env/tasks"
files = sorted(os.listdir(tasks_dir))
print(f"\n✅ Available task files ({len(files)} total):")
for f in files:
    size = os.path.getsize(os.path.join(tasks_dir, f))
    print(f"   {f} ({size} bytes)")

## 4. Create Training Dataset (~128 Samples)

In [ ]:
from datasets import Dataset
import os

# System prompt (same as Colab)
SYSTEM_PROMPT = """You are an expert data cleaning agent. Clean messy CSV files by:
1. Reading the file with read_file(path)
2. Using run_python(code) to clean with pandas (remove duplicates, fix missing values, normalize dates, trim whitespace, etc.)
3. Submitting with submit_cleaned_file(path='cleaned_output.csv')

Always save cleaned data with: df.to_csv('cleaned_output.csv', index=False)"""

# Training prompts covering different domains and difficulty levels
prompts = [
    # Easy - customer data
    f"{SYSTEM_PROMPT}\n\nTask: Clean customer dataset. Remove duplicates and handle missing values. File: easy_messy.csv",
    f"{SYSTEM_PROMPT}\n\nTask: Clean this dataset with duplicate rows and missing emails. File: easy_messy.csv",
    
    # Medium - sales records  
    f"{SYSTEM_PROMPT}\n\nTask: Clean sales records. Fix: inconsistent dates, mixed casing, invalid emails, extra whitespace. File: medium_messy.csv",
    f"{SYSTEM_PROMPT}\n\nTask: Normalize dates to YYYY-MM-DD, title case names, validate emails, convert amounts to float. File: medium_messy.csv",
    
    # Hard - product inventory
    f"{SYSTEM_PROMPT}\n\nTask: Clean product inventory. Fix: missing IDs, negative quantities, invalid prices, inconsistent formats. File: hard_messy.csv",
    f"{SYSTEM_PROMPT}\n\nTask: Generate missing IDs, clamp negatives to 0, normalize all data. File: hard_messy.csv",
]

# Add domain-specific prompts if files exist (same as Colab)
domain_prompts = [
    # HR domain
    ("hr_messy.csv", "Clean HR employee records. Fix: name casing, salary formats ($, USD), date formats, email issues, missing values."),
    ("hr_messy.csv", "Standardize employee data: title case names, numeric salaries, YYYY-MM-DD dates, valid lowercase emails."),
    
    # Healthcare domain
    ("healthcare_messy.csv", "Clean patient records. Fix: name casing, condition casing, date formats, doctor name formats, status casing."),
    ("healthcare_messy.csv", "Normalize healthcare data: title case all names, standardize dates, fill missing insurance IDs."),
    
    # Finance domain
    ("finance_messy.csv", "Clean financial transactions. Fix: merchant casing, amount formats ($, USD), date formats, category casing."),
    ("finance_messy.csv", "Standardize transaction data: title case merchants, numeric amounts, YYYY-MM-DD dates."),
    
    # Logistics domain
    ("logistics_messy.csv", "Clean shipping records. Fix: origin/destination casing, carrier names, date formats, status casing."),
    ("logistics_messy.csv", "Normalize logistics data: title case locations, standardize dates, fix weight formats."),
    
    # Education domain
    ("education_messy.csv", "Clean student records. Fix: name casing, subject casing, date formats, status casing, missing credits."),
    ("education_messy.csv", "Standardize grade records: title case names/subjects, YYYY-MM-DD dates, fill missing credits."),
]

tasks_dir = "data_clean_env/tasks"
for filename, task_desc in domain_prompts:
    if os.path.exists(os.path.join(tasks_dir, filename)):
        prompts.append(f"{SYSTEM_PROMPT}\n\nTask: {task_desc} File: {filename}")

# Repeat prompts for MORE training samples (~128 like Colab)
train_dataset = Dataset.from_dict({"prompt": prompts * 8})  # ~128 samples
print(f"✅ Created training dataset with {len(train_dataset)} samples")
print(f"   Unique prompts: {len(prompts)}")
print(f"   Domains covered: easy, medium, hard + HR, healthcare, finance, logistics, education")

## 5. Define Reward Function

In [ ]:
def reward_fn(completions, **kwargs):
    """Heuristic reward function for GRPO training.
    
    Rewards proper tool usage and cleaning operations.
    No environment server needed - fast and stable.
    """
    rewards = []
    
    for completion in completions:
        if isinstance(completion, list):
            text = " ".join([str(t) for t in completion])
        else:
            text = str(completion)
        
        reward = 0.0
        
        # Step 1: Read file
        if "read_file" in text:
            reward += 0.15
        
        # Step 2: Use pandas to clean
        if "run_python" in text:
            reward += 0.20
            
            # Bonus for cleaning operations
            cleaning_ops = [
                "drop_duplicates", "fillna", "dropna", "strip",
                "to_datetime", "str.lower", "str.upper", "str.title",
                "replace", "astype", "rename", "apply"
            ]
            ops_found = sum(1 for op in cleaning_ops if op in text)
            reward += min(0.15, ops_found * 0.03)
        
        # Step 3: Save cleaned file
        if "to_csv" in text:
            reward += 0.15
            if "cleaned" in text.lower():
                reward += 0.1
        
        # Step 4: Submit
        if "submit_cleaned_file" in text:
            reward += 0.2
            if "cleaned_output.csv" in text:
                reward += 0.1
        
        # Cap at 1.0
        rewards.append(min(1.0, reward))
    
    return rewards

print("✅ Reward function defined")

## 6. Load Model with 4-bit Quantization (Memory-Optimized)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import gc

# Clear memory before loading
gc.collect()
torch.cuda.empty_cache()

print(f"Loading model: {MODEL_ID}")
print(f"GPU memory before: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# 4-bit quantization for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model with low_cpu_mem_usage for Kaggle
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,  # Important for Kaggle!
)

print(f"\n✅ Model loaded: {MODEL_ID}")
print(f"   Parameters: {model.num_parameters() / 1e9:.2f}B")
print(f"   GPU Memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"   GPU Memory free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB")

## 7. Configure GRPO Trainer (Memory-Optimized)

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from peft import LoraConfig

# ============================================
# MEMORY-OPTIMIZED LoRA (fewer target modules)
# ============================================
lora_config = LoraConfig(
    r=8,              # REDUCED from 16 (saves memory)
    lora_alpha=16,    # REDUCED from 32
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],  # REDUCED (only essential projections)
    task_type="CAUSAL_LM",
)

# GRPO training config
training_args = GRPOConfig(
    output_dir=CONFIG["output_dir"],
    learning_rate=CONFIG["learning_rate"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    num_generations=CONFIG["num_generations"],
    max_completion_length=CONFIG["max_completion_length"],
    temperature=CONFIG["temperature"],
    logging_steps=CONFIG["logging_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=1,  # Save less checkpoints
    report_to="none",
    bf16=True,
    gradient_checkpointing=True,
    warmup_steps=5,
)

# Initialize trainer
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_fn,
    args=training_args,
    train_dataset=train_dataset,
    peft_config=lora_config,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ GRPO Trainer ready (memory-optimized)")
print(f"   Trainable params: {trainable / 1e6:.1f}M")
print(f"   Training samples: {len(train_dataset)}")
print(f"   LoRA rank: {lora_config.r} (reduced for memory)")
print(f"   Target modules: {lora_config.target_modules}")
print(f"   GPU Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 8. Train! 🚀

In [ ]:
print("=" * 50)
print("🚀 STARTING GRPO TRAINING")
print("=" * 50)
print(f"Model: {MODEL_ID}")
print(f"Samples: {len(train_dataset)}")
print(f"Epochs: {CONFIG['num_train_epochs']}")
print(f"Batch: {CONFIG['per_device_train_batch_size']} × {CONFIG['gradient_accumulation_steps']} accum")
print(f"Generations: {CONFIG['num_generations']} per prompt")
print()

# Train
train_result = trainer.train()

# Save
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])

print()
print("=" * 50)
print("✅ TRAINING COMPLETE!")
print("=" * 50)
print(f"Model saved to: {CONFIG['output_dir']}")
print(f"Final loss: {train_result.training_loss:.4f}")

## 9. Test the Trained Model

In [ ]:
from transformers import pipeline

# Merge LoRA weights
print("Merging LoRA weights...")
merged_model = trainer.model.merge_and_unload()

# Create pipeline
pipe = pipeline(
    "text-generation",
    model=merged_model,
    tokenizer=tokenizer,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

# Test prompt
test_prompt = f"""{SYSTEM_PROMPT}

Task: Clean customer data with duplicates and missing emails. File: customers_messy.csv"""

print("\n📝 Testing trained model...")
output = pipe(
    test_prompt,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
    return_full_text=False,
)

print("\n" + "=" * 50)
print("MODEL OUTPUT:")
print("=" * 50)
print(output[0]["generated_text"])

## 10. Push to HuggingFace Hub (Optional)

In [ ]:
# Uncomment and fill in to push your model

# HF_USERNAME = "your-username"  # Your HuggingFace username
# MODEL_NAME = "dataclean-agent-grpo"

# if HF_TOKEN:
#     merged_model.push_to_hub(f"{HF_USERNAME}/{MODEL_NAME}", token=HF_TOKEN)
#     tokenizer.push_to_hub(f"{HF_USERNAME}/{MODEL_NAME}", token=HF_TOKEN)
#     print(f"✅ Pushed to https://huggingface.co/{HF_USERNAME}/{MODEL_NAME}")
# else:
#     print("❌ Set HF_TOKEN to push model")

## 11. Download Model (to use locally)

In [ ]:
# Zip the checkpoint for download
import shutil

output_zip = f"{CONFIG['output_dir']}.zip"
shutil.make_archive(CONFIG['output_dir'], 'zip', CONFIG['output_dir'])
print(f"✅ Model zipped: {output_zip}")
print(f"   Download from Kaggle Output tab")